In [1]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter
from typing import List
from tqdm import tqdm
import joblib
import weaviate
import re
from weaviate.util import generate_uuid5
from pprint import pprint
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "valence", "tempo"
]

In [3]:
df = pd.read_csv("SpotifyFeatures.csv")

df = df.dropna(subset=AUDIO_FEATURES)
scaler = StandardScaler()
df[AUDIO_FEATURES] = scaler.fit_transform(df[AUDIO_FEATURES])

In [4]:
df.head()

,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
0,Movie,Henri Salvador,C'est beau de faire un Show,0BRjO6ga9RKCKjfDqeFgWV,0,0.683376,-0.890935,99373,1.286908,-0.489819,C#,0.660661,1.290703,Major,-0.367970,1.595607,4/4,1.380744
1,Movie,Martin & les fées,Perdu d'avance (par Gad Elmaleh),0BjC1NfoEOOusryehmNudP,1,-0.345467,0.191994,137373,0.630249,-0.489819,F#,-0.322835,0.668683,Minor,-0.183082,1.823253,4/4,1.388435
2,Movie,Joseph Williams,Don't Let Me Be Lonely Tonight,0CoSDzoNIKCRs124s9uTVy,3,1.644570,0.585296,170267,-1.669954,-0.489819,C,-0.564927,-0.718402,Minor,-0.455832,-0.588326,5/4,-0.334212
3,Movie,Henri Salvador,Dis-moi Monsieur Gordon Cooper,0Gc6TVm52BwZD07Ki6tIvf,0,0.942701,-1.693703,152427,-0.929789,-0.489819,C#,-0.587623,-0.434817,Major,-0.438044,1.750597,4/4,-0.876384
4,Movie,Fabien Nataf,Ouverture,0IuslXpMROHdEPvSl1fTQK,4,1.638932,-1.203422,82625,-1.313157,-0.083566,F,-0.065613,-1.930601,Major,-0.405163,0.741433,4/4,-0.249618


In [5]:
client = weaviate.connect_to_local()

In [6]:
# Delete the collection in case it exists
if client.collections.exists("Song"):
    client.collections.delete("Song")
    

In [7]:
client.collections.create(
    name="Song",
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="artist", data_type=DataType.TEXT),
        Property(name="genre", data_type=DataType.TEXT),
        Property(name="language", data_type=DataType.TEXT),
        Property(name="mood", data_type=DataType.TEXT),
    ]
)

/Users/ujjwalkatyal/anaconda3/lib/python3.11/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


In [8]:
songs = client.collections.get("Song")

with songs.batch.dynamic() as batch:
    for _, row in tqdm(df.iterrows(), total=len(df)):
        batch.add_object(
            properties={
                "title": row["track_name"],
                "artist": row["artist_name"],
                "genre": row["genre"],
                "language": row.get("language", "unknown"),
                "mood": "happy" if row["valence"] > 0 else "sad"
            },
            vector=row[AUDIO_FEATURES].tolist()
        )

100%|█████████████████████████████████| 232725/232725 [01:27<00:00, 2656.13it/s]


In [9]:
from weaviate.collections.classes.filters import Filter

def recommend_songs(song_title, k=5):
    song = songs.query.fetch_objects(
        filters=Filter.by_property("title").equal(song_title),
        limit=1,
        include_vector=True
    )

    if not song.objects:
        return "Song not found"

    vector = song.objects[0].vector["default"]

    results = songs.query.near_vector(
        near_vector=vector,
        filters=Filter.by_property("genre").equal("Pop"),
        limit=k + 1
    )

    return [
        {
            "title": r.properties["title"],
            "artist": r.properties["artist"],
            "genre": r.properties.get("genre", "unknown")
        }
        for r in results.objects[1:]  # skip the query song itself
    ]

In [10]:
recommend_songs("Shape of You", k=5)

[{'title': 'A Dios Le Pido', 'artist': 'Juanes', 'genre': 'Pop'},
 {'title': 'Bailando - English Version',
  'artist': 'Enrique Iglesias',
  'genre': 'Pop'},
 {'title': 'Popular Song', 'artist': 'MIKA', 'genre': 'Pop'},
 {'title': 'Kiss It Better', 'artist': 'Rihanna', 'genre': 'Pop'},
 {'title': 'Nancy Mulligan', 'artist': 'Ed Sheeran', 'genre': 'Pop'}]